# SLAM-ASR: An Embarrassingly Simple Approach for LLM with Strong ASR Capacity

[![arXiv](https://img.shields.io/badge/arXiv-2402.08846-b31b1b.svg)](https://arxiv.org/abs/2402.08846)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gokhanersoy/slam-asr-reimplement/blob/main/slam_asr_colab.ipynb)

Bu notebook, **arXiv:2402.08846** çalışmasının Google Colab (ücretsiz T4 veya A100 GPU) üzerinde uçtan uca çalıştırılması için hazırlanmıştır.

### 🌟 Temel Mimari Prensipleri
1. **Speech Foundation Encoder (FROZEN):** HuBERT, WavLM veya Whisper ağırlıkları dondurulur.
2. **Frame Downsampler ($k=5$):** 50 Hz olan konuşma öznitelikleri, ardışık 5 çerçevenin birleştirilmesiyle **10 Hz'e** indirgenir.
3. **Trainable Projector (TRAINABLE):** Yalnızca tek katmanlı lineer katman (veya 2 katmanlı MLP) eğitilir (~18M - 21M parametre).
4. **Causal LLM Backbone (FROZEN):** Vicuna-7B veya TinyLlama dondurulmuş olarak kullanılır.
5. **Format:** `USER: <S> <Prompt> ASSISTANT: <Transcript><eos>` (Kayıp yalnızca hedef metin tokenları üzerinden hesaplanır).

## 1. GPU ve Donanım Kontrolü

In [1]:
# GPU kullanılabilirliğini kontrol edin ve donanımınıza en uygun hiperparametreleri otomatik belirleyin
!nvidia-smi

try:
    from slam_asr.utils.device_utils import get_device_specs, get_recommended_hyperparameters, format_device_summary
    specs = get_device_specs()
    rec = get_recommended_hyperparameters()
    print(format_device_summary(
        specs=specs,
        rec=rec,
        model_cfg={"torch_dtype": rec["torch_dtype"], "load_in_4bit": rec["load_in_4bit"]},
        data_cfg={"batch_size": rec["batch_size"], "num_workers": rec["num_workers"]},
        train_cfg={"gradient_accumulation_steps": rec["gradient_accumulation_steps"], "bf16": rec["bf16"], "fp16": rec["fp16"]},
    ))
except Exception as e:
    import torch
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"Device Name: {torch.cuda.get_device_name(0)}")
        print(f"Device Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


Mon Oct 12 21:19:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Proje Dosyalarını Çekme ve Bağımlılıkları Yükleme

Proje dosyalarını Colab'a aktarmak için aşağıdaki **iki yöntemden birini** kullanabilirsiniz:

- **Yöntem 1 (Önerilen):** GitHub deposundan doğrudan klonlamak (`git clone`).
- **Yöntem 2:** Bilgisayarınızdaki `slam-asr-reimplement.zip` dosyasını Colab'ın sol dosya paneline sürükleyip açmak (`unzip`).

In [2]:
import os

# Eğer çalışma dizininde slam_asr klasörü yoksa GitHub'dan klonla
if not os.path.exists("slam_asr") and not os.path.exists("slam-asr-reimplement/slam_asr"):
    print("[*] Proje GitHub'dan klonlanıyor...")
    !git clone https://github.com/gokhanersoy/slam-asr-reimplement.git
    %cd slam-asr-reimplement
elif os.path.exists("slam-asr-reimplement"):
    %cd slam-asr-reimplement
else:
    print("[*] Proje dosyaları zaten aktif dizinde mevcut.")

# Gerekli kütüphaneleri requirements.txt üzerinden yükleyin
!pip install -q -r requirements.txt

[*] Proje GitHub'dan klonlanıyor...
Cloning into 'slam-asr-reimplement'...
remote: Enumerating objects: 770, done.
remote: Counting objects: 100% (770/770), done.
remote: Compressing objects: 100% (673/673), done.
remote: Total 770 (delta 123), reused 727 (delta 80), pack-reused 0 (from 0)
Receiving objects: 100% (770/770), 2.99 MiB | 8.23 MiB/s, done.
Resolving deltas: 100% (123/123), done.
/content/slam-asr-reimplement
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 140.6 MB/s eta 0:00:00


## 3. Kod Dosyalarının Doğrulanması ve Birim Testler

In [3]:
# Testleri koşturarak modüllerin (Projector, Collator, Metrics, Model) doğru çalıştığını teyit edin
!python run_tests.py

test_variable_length_waveform_collation (test_collator.TestCollator.test_variable_length_waveform_collation) ... ok
test_whisper_mel_collation (test_collator.TestCollator.test_whisper_mel_collation) ... ok
test_compute_cer (test_metrics.TestMetrics.test_compute_cer) ... ok
test_compute_wer (test_metrics.TestMetrics.test_compute_wer) ... ok
test_levenshtein_distance (test_metrics.TestMetrics.test_levenshtein_distance) ... ok
test_text_normalization (test_metrics.TestMetrics.test_text_normalization) ... ok
test_downsampler_discard_remainder (test_projector.TestProjector.test_downsampler_discard_remainder) ... ok
test_downsampler_pad_remainder (test_projector.TestProjector.test_downsampler_pad_remainder) ... ok
test_linear_projector (test_projector.TestProjector.test_linear_projector) ... ok
test_mlp_projector (test_projector.TestProjector.test_mlp_projector) ... ok
test_speech_projector_end_to_end (test_projector.TestProjector.test_speech_projector_end_to_end) ... ok
test_forward_pass_an

## 4. Hızlı Doğrulama: Sentetik Veri ile Test Eğitimi
Sistemin forward-backward döngüsünün, gradyan izolasyonunun ve erken durdurma mekanizmasının çalıştığını saniyeler içinde görmek için sentetik ses verisi üretip test edebilirsiniz.

In [4]:
# 1. Sentetik toy verisini üretin
!python scripts/create_toy_dataset.py

# 2. Hızlı debug eğitimi başlatın (T4 GPU üzerinde saniyeler sürer)
!python train.py --config config/train_toy_debug.yaml --max_steps 20


[*] Generated toy dataset:
    Train: /content/slam-asr-reimplement/data/sample_train.jsonl (4 samples)
    Val:   /content/slam-asr-reimplement/data/sample_val.jsonl (1 samples)
    Audio: /content/slam-asr-reimplement/data/toy_audio
[*] Initializing SLAM-ASR with config: config/train_toy_debug.yaml
config.json: 100% 1.39k/1.39k [00:00<00:00, 8.44MB/s]

pytorch_model.bin: downloading bytes:   6% 24.0M/378M [00:00<00:10, 33.7MB/s]
pytorch_model.bin: downloading bytes:  15% 56.2M/378M [00:00<00:03, 82.0MB/s, 2.35MB/s  ]
pytorch_model.bin: reconstructing file:  16% 61.7M/378M [00:00<00:04, 73.3MB/s, 4.14MB/s  ]
pytorch_model.bin: downloading bytes:  28% 106M/378M [00:01<00:01, 136MB/s, 7.30MB/s  ]  
pytorch_model.bin: downloading bytes:  42% 158M/378M [00:01<00:01, 181MB/s, 12.5MB/s  ]
pytorch_model.bin: downloading bytes:  56% 212M/378M [00:01<00:00, 215MB/s, 16.6MB/s  ]
pytorch_model.bin: downloading bytes: 100% 254M/254M [00:01<00:00, 153MB/s, 23.5MB/s  ]
pytorch_model.bin: reconstruc

## 5. Gerçek LibriSpeech Verisi Hazırlığı (Hugging Face Üzerinden)
Hugging Face `openslr/librispeech_asr` veri setinden küçük bir dilim indirip manifestoları otomatik hazırlayalım.

In [5]:
import os
import json
import soundfile as sf
from datasets import load_dataset
from tqdm import tqdm

os.makedirs("data/colab_audio", exist_ok=True)

def prepare_hf_split(split_name, output_path, max_samples=3000):
    print(f"[*] Downloading '{split_name}' from openslr/librispeech_asr (max {max_samples} samples)...")
    try:
        ds = load_dataset("openslr/librispeech_asr", "clean", split=split_name, streaming=True)
    except Exception:
        ds = load_dataset("distil-whisper/librispeech_asr", "clean", split=split_name, streaming=True)

    records = []
    for i, item in enumerate(tqdm(ds, total=max_samples)):
        if max_samples and i >= max_samples:
            break
        key = item.get("id") or f"sample_{i}"
        audio_info = item["audio"]
        wav_path = os.path.abspath(f"data/colab_audio/{key}.wav")
        if not os.path.exists(wav_path):
            sf.write(wav_path, audio_info["array"], audio_info["sampling_rate"])

        records.append({
            "key": str(key),
            "source": wav_path,
            "target": item["text"],
            "prompt": "Transcribe speech to text."
        })

    with open(output_path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")
    print(f"[*] Saved {len(records)} samples to {output_path}")

# Makaledeki doğruluk oranına ulaşmak için eğitim verisi miktarını artırıyoruz (~15 saat ses)
# Tam LibriSpeech-100h (28,539 örnek) için max_samples=None yapabilirsiniz.
MAX_TRAIN_SAMPLES = 3000
MAX_VAL_SAMPLES = 200

prepare_hf_split("train.100", "data/colab_train.jsonl", max_samples=MAX_TRAIN_SAMPLES)
prepare_hf_split("validation", "data/colab_val.jsonl", max_samples=MAX_VAL_SAMPLES)


[*] Downloading 'train.100' from openslr/librispeech_asr (max 3000 samples)...


README.md:   0%|          | 0.00/11.0k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

100%|██████████| 3000/3000 [00:25<00:00, 118.90it/s]


[*] Saved 3000 samples to data/colab_train.jsonl
[*] Downloading 'validation' from openslr/librispeech_asr (max 200 samples)...


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

100%|██████████| 200/200 [00:01<00:00, 103.08it/s]


[*] Saved 200 samples to data/colab_val.jsonl


## 6. Donanıma Göre Otomatik Uyarlanan SLAM-ASR Eğitimi (Hardware-Adaptive Training)

Seçtiğiniz donanım hızlandırıcı ne olursa olsun (**G4, T4, L4, A100, H100 veya CPU**), `config/hardware_adaptive_vicuna.yaml` dosyası:
- **VRAM Kapasitesi** ve **Hesaplama Mimarisi (Ampere, Hopper, Ada Lovelace, Turing)** analizini dinamik olarak yapar.
- **Precision:** Ampere/Ada/Hopper için yerel `bfloat16`, Turing T4 için `float16`, CPU için `float32` seçer.
- **Kuantizasyon:** 34 GB ve üzeri kartlarda (A100/H100) makaledeki gibi kuantizasyonsuz tam 16-bit; T4/L4 için 4-bit (NF4) seçer.
- **Batch Size & Gradient Accumulation:** Efektif batch boyutunu makaledeki gibi 16 olarak optimize eder.


In [6]:
# Donanıma göre otomatik optimize edilen SLAM-ASR eğitimi (Vicuna-7B + HuBERT-Large)
!python train.py \
    --config config/hardware_adaptive_vicuna.yaml \
    --train_manifest data/colab_train.jsonl \
    --val_manifest data/colab_val.jsonl \
    --output_dir ./outputs/slam_asr_adaptive_vicuna \
    --max_steps 3000


[*] Initializing SLAM-ASR with config: config/colab_t4_vicuna_4bit.yaml
config.json: 100% 1.38k/1.38k [00:00<00:00, 9.10MB/s]

pytorch_model.bin: downloading bytes:   9% 114M/1.26G [00:00<00:06, 177MB/s, 5.85MB/s  ]  
pytorch_model.bin: downloading bytes:  20% 249M/1.26G [00:01<00:03, 322MB/s, 17.8MB/s  ]
pytorch_model.bin: downloading bytes:  29% 367M/1.26G [00:01<00:02, 433MB/s, 28.9MB/s  ]
pytorch_model.bin: downloading bytes:  37% 468M/1.26G [00:01<00:01, 464MB/s, 38.7MB/s  ]
pytorch_model.bin: downloading bytes:  48% 605M/1.26G [00:01<00:01, 553MB/s, 49.9MB/s  ]
pytorch_model.bin: downloading bytes:  55% 699M/1.26G [00:02<00:01, 518MB/s, 58.9MB/s  ]
pytorch_model.bin: downloading bytes:  61% 768M/1.26G [00:02<00:01, 455MB/s, 67.7MB/s  ]
pytorch_model.bin: downloading bytes:  66% 832M/1.26G [00:02<00:01, 407MB/s, 71.7MB/s  ]
pytorch_model.bin: downloading bytes: 100% 848M/848M [00:02<00:00, 311MB/s, 74.4MB/s  ]
pytorch_model.bin: reconstructing file: 100% 1.26G/1.26G [00:02<00:00, 

## 7. Model Değerlendirmesi (WER ve CER Hesaplama)

In [7]:
# Donanıma göre optimize edilmiş model üzerinde test manifestosunu değerlendirin (Beam size = 4)
!python evaluate.py \
    --config config/hardware_adaptive_vicuna.yaml \
    --checkpoint outputs/slam_asr_adaptive_vicuna/best_checkpoint \
    --manifest data/colab_val.jsonl \
    --beam_size 4


[*] Loading model on cuda...
Loading weights: 100% 422/422 [00:00<00:00, 66679.08it/s]
[transformers] HubertModel LOAD REPORT from: facebook/hubert-large-ls960-ft
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [00:01<00:00, 259.31it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[*] Loading projector weights from: outputs/slam_asr_colab_vicuna/best_checkpoint/projector.pt
[*] Evaluating 200 samples with beam_size=4...
Evaluating:   0% 0/50 [00:00<?, ?it/s]/content/slam-asr-reimplement/evaluate.py:113: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please us

## 8. İnteraktif Çıkarım (Ses Dinleme ve Transcribe)

In [8]:
import json
import os
import IPython.display as ipd

# Gerçek insan sesi örneğini doğrulama manifestosundan çekelim
audio_path = "data/toy_audio/toy_1.wav"
ref_text = ""
if os.path.exists("data/colab_val.jsonl"):
    with open("data/colab_val.jsonl", "r") as f:
        first_item = json.loads(f.readline())
        audio_path = first_item["source"]
        ref_text = first_item["target"]
    print(f"[*] Gerçek LibriSpeech insan sesi yüklendi: {audio_path}")
    print(f"[*] Gerçek Metin (Ground Truth): {ref_text}\n")
else:
    print("[*] data/colab_val.jsonl bulunamadı, toy ses kullanılıyor.")

# Sesi dinleyin (Gerçek insan konuşması):
ipd.display(ipd.Audio(audio_path))

# Donanıma uyarlanmış SLAM-ASR modeliyle sesi yazıya dökün
!python inference.py --audio "$audio_path" --config config/hardware_adaptive_vicuna.yaml --checkpoint outputs/slam_asr_adaptive_vicuna/best_checkpoint/projector.pt --prompt "Transcribe speech to text." --beam_size 4


[*] Gerçek LibriSpeech insan sesi yüklendi: /content/slam-asr-reimplement/data/colab_audio/2277-149896-0000.wav
[*] Gerçek Metin (Ground Truth): HE WAS IN A FEVERED STATE OF MIND OWING TO THE BLIGHT HIS WIFE'S ACTION THREATENED TO CAST UPON HIS ENTIRE FUTURE



[*] Loading SLAM-ASR on cuda...
Loading weights: 100% 422/422 [00:00<00:00, 69550.72it/s]
[transformers] HubertModel LOAD REPORT from: facebook/hubert-large-ls960-ft
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [00:01<00:00, 263.62it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[*] Loading projector weights from: outputs/slam_asr_colab_vicuna/best_checkpoint/projector.pt
[*] Processing audio: /content/slam-asr-reimplement/data/colab_audio/2277-149896-0000.wav
[*] Transcribing (beam_size=4)...
/content/slam-asr-reimplement/inference.py:82: FutureWarning: `torch.cuda.amp.a